<a href="https://colab.research.google.com/github/Rok1375/LINEAR/blob/main/LLM_inSTRCUTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Python libraries
!pip install -q playwright reportlab beautifulsoup4 lxml nest-asyncio

# Install system-level Linux dependencies required by Chromium
!playwright install-deps chromium

# Install the browser engine
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.7 MB/s eta 0:00:00
Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,061 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import sys
import subprocess

# ==========================================
# 1. IMMEDIATE USER INPUT (No Waiting!)
# ==========================================
print("="*60)
print(" 🚀 AI-DRIVEN WEBSITE EXTRACTOR & PROMPT GENERATOR ")
print("="*60)

# Ask for URL immediately before loading heavy background libraries
target_url = input("\n🌐 1. Enter the Target URL (e.g., linear.app): ").strip()
if not target_url:
    print("❌ Error: No URL provided. Exiting...")
    sys.exit(1)
if not target_url.startswith(('http://', 'https://')):
    target_url = 'https://' + target_url

# Ask for Device Viewport
print("\n📱 2. Select Device Viewport:")
print("  [1] Desktop (1920x1080) - Default")
print("  [2] Mobile iPhone (375x812)")
device_choice = input("Choice (1/2): ").strip()

# Ask for Color Theme
print("\n🎨 3. Force Color Scheme:")
print("  [1] Dark Mode - Default")
print("  [2] Light Mode")
theme_choice = input("Choice (1/2): ").strip()

# Apply choices
viewport = {"width": 375, "height": 812} if device_choice == "2" else {"width": 1920, "height": 1080}
device_name = "Mobile" if device_choice == "2" else "Desktop"
color_scheme = "light" if theme_choice == "2" else "dark"

print(f"\n✅ Configuration Saved: Extracting {target_url} on {device_name} ({color_scheme.upper()} mode).")
print("⏳ Initializing environment and verifying dependencies... (Please wait)\n")

# ==========================================
# 2. AUTO-DEPENDENCY INSTALLER
# ==========================================
def install_dependencies():
    required_packages = ["playwright", "reportlab", "beautifulsoup4", "lxml", "nest-asyncio"]
    try:
        import playwright
        import reportlab
        import nest_asyncio
        from bs4 import BeautifulSoup
    except ImportError:
        print("📦 Missing dependencies detected. Installing them invisibly now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required_packages])
        print("🌐 Installing Playwright Chromium browser...")
        subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
        print("✅ Environment ready!\n")

install_dependencies()

# NOW we import the heavy libraries safely
import json
import shutil
import asyncio
import urllib.parse
import hashlib
from html import escape
from playwright.async_api import async_playwright
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from bs4 import BeautifulSoup
import nest_asyncio

nest_asyncio.apply()

# ==========================================
# DIRECTORY SETUP
# ==========================================
BASE_DIR = os.getcwd()
WORKSPACE_DIR = os.path.join(BASE_DIR, "Publish_Ready_Workspace")
RAW_DIR = os.path.join(BASE_DIR, "raw_extracted_data")
ZIP_FILENAME = "Publish_Ready_Website_Data"

# ==========================================
# 3. DEEP EXTRACTION ENGINE
# ==========================================
async def extract_site(url):
    print(f"\n🚀 PHASE 1: Launching Deep Extractor for {url}")

    if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(os.path.join(RAW_DIR, "network_assets"), exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-setuid-sandbox'])
        context = await browser.new_context(
            viewport=viewport,
            color_scheme=color_scheme,
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        async def handle_response(response):
            try:
                if response.ok and response.request.resource_type in ["document", "stylesheet", "script", "image", "font", "media"]:
                    req_url = response.url
                    if req_url.startswith(("data:", "blob:")): return

                    parsed_url = urllib.parse.urlparse(req_url)
                    path = parsed_url.path if parsed_url.path else '/index.html'
                    clean_path = path.split('?')[0].lstrip('/')
                    if clean_path == "" or clean_path.endswith('/'):
                        clean_path = os.path.join(clean_path, "index.html")

                    file_path = os.path.join(RAW_DIR, "network_assets", parsed_url.netloc, clean_path)
                    os.makedirs(os.path.dirname(file_path), exist_ok=True)

                    with open(file_path, 'wb') as f: f.write(await response.body())
            except Exception: pass

        page.on("response", handle_response)

        print(" -> Loading page and intercepting internal architecture...")
        try: await page.goto(url, wait_until="networkidle", timeout=60000)
        except Exception: pass

        print(" -> Scrolling page to trigger lazy-loaded motion UI (GSAP/Framer)...")
        await page.evaluate("""async () => {
            await new Promise((resolve) => {
                let totalHeight = 0; const distance = 300;
                const timer = setInterval(() => {
                    window.scrollBy(0, distance); totalHeight += distance;
                    if(totalHeight >= document.body.scrollHeight || totalHeight > 15000) { clearInterval(timer); resolve(); }
                }, 250);
            });
        }""")
        await page.wait_for_timeout(3000)

        full_html = await page.content()
        with open(os.path.join(RAW_DIR, "full_rendered_page.html"), "w", encoding="utf-8") as f:
            f.write(full_html)

        print(" -> Segmenting DOM layout into AI components...")
        sections_data = await page.evaluate("""() => {
            const sections = document.querySelectorAll('header, section, footer, main > div');
            const data = [];
            sections.forEach((sec, idx) => {
                const html = sec.outerHTML;
                if(html.length < 250) return;
                let type = 'Generic_Section';
                const idClass = (sec.id + " " + sec.className).toLowerCase();

                if(sec.tagName === 'HEADER' || idClass.includes('hero') || idClass.includes('nav') || idClass.includes('banner')) type = 'Header_Hero';
                else if(idClass.includes('service') || idClass.includes('feature')) type = 'Services';
                else if(idClass.includes('about') || idClass.includes('team')) type = 'About';
                else if(idClass.includes('contact') || sec.tagName === 'FOOTER') type = 'Footer_Contact';
                else if(idClass.includes('portfolio') || idClass.includes('work') || idClass.includes('project')) type = 'Portfolio';

                data.push({ type: type, id: sec.id || `section-${idx}`, classes: sec.className, html: html });
            });
            return data;
        }""")

        with open(os.path.join(RAW_DIR, "sections_data.json"), "w", encoding="utf-8") as f:
            json.dump(sections_data, f)

        print(f"✅ Extraction complete! Found {len(sections_data)} major UI components.")
        await browser.close()

# ==========================================
# 4. DATA ORGANIZER & MASTER PDF BUILDER
# ==========================================
def organize_data(url):
    print("\n📂 PHASE 2: Organizing Data & Generating Master PDF Blueprint...")
    if os.path.exists(WORKSPACE_DIR): shutil.rmtree(WORKSPACE_DIR)

    folders = ["AI_Component_Prompts", "Global_Assets/Images", "Global_Assets/CSS", "Global_Assets/JS", "Global_Assets/Fonts", "Full_Playable_Site"]
    for f in folders: os.makedirs(os.path.join(WORKSPACE_DIR, f), exist_ok=True)

    sections_path = os.path.join(RAW_DIR, "sections_data.json")
    if os.path.exists(sections_path):
        with open(sections_path, "r", encoding="utf-8") as f: sections = json.load(f)

        pdf_path = os.path.join(WORKSPACE_DIR, "LLM_Recreation_Blueprint.pdf")
        doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40)
        styles = getSampleStyleSheet()

        title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=20, textColor=HexColor('#1a202c'), spaceAfter=15)
        h2_style = ParagraphStyle('H2Style', parent=styles['Heading2'], fontSize=14, textColor=HexColor('#2b6cb0'), spaceTop=20, spaceAfter=10)
        normal_style = ParagraphStyle('NormalStyle', parent=styles['Normal'], fontSize=10, textColor=HexColor('#4a5568'), leading=14)
        prompt_style = ParagraphStyle('PromptStyle', parent=styles['Normal'], fontSize=10, leading=15, textColor=HexColor('#1e40af'), backColor=HexColor('#eff6ff'), borderPadding=10, borderColor=HexColor('#bfdbfe'), borderWidth=1, borderRadius=5, spaceAfter=15)
        code_style = ParagraphStyle('CodeStyle', parent=styles['Normal'], fontName='Courier', fontSize=7, leading=9, textColor=HexColor('#c53030'), backColor=HexColor('#f7fafc'), borderPadding=10, borderColor=HexColor('#e2e8f0'), borderWidth=1, borderRadius=5, wordWrap='CJK')

        story = []
        story.append(Paragraph("Master UI Recreation Blueprint", title_style))
        story.append(Paragraph(f"<b>Target:</b> {url} | <b>Device:</b> {device_name} | <b>Theme:</b> {color_scheme.upper()}", normal_style))
        story.append(Spacer(1, 20))
        story.append(Paragraph("🤖 Instructions for Large Language Models (Claude 3.5, GPT-4o, Cursor)", h2_style))
        story.append(Paragraph(f"""You are acting as an <b>Expert Frontend Engineer & Motion UI Developer</b>. Recreate this website block by block using modern HTML5 and Tailwind CSS. Ensure it is optimized for a {device_name} layout. Infer grid/flex layouts, recreate implied GSAP/Framer animations based on classes, and maintain exact DOM structures.""", normal_style))
        story.append(PageBreak())

        for idx, sec in enumerate(sections, 1):
            folder_name = f"{idx:02d}_{sec['type']}"
            comp_dir = os.path.join(WORKSPACE_DIR, "AI_Component_Prompts", folder_name)
            os.makedirs(comp_dir, exist_ok=True)
            with open(os.path.join(comp_dir, "source_code.html"), "w", encoding="utf-8") as f: f.write(sec['html'])

            story.append(Paragraph(f"Part {idx:02d}: {sec['type']}", title_style))
            story.append(Paragraph(f"<b>Target ID:</b> {sec['id']} <br/><b>Target Classes:</b> {sec['classes']}", normal_style))
            story.append(Spacer(1, 10))
            story.append(Paragraph(f"<b>LLM Action Prompt:</b><br/>Recreate this '{sec['type']}' component. Maintain the exact nesting below. Synthesize implied UI motions and Tailwind styling. Output clean code.", prompt_style))

            raw_html = sec['html'][:4000] + "\n\n" if len(sec['html']) > 4000 else sec['html']
            formatted_html = escape(raw_html).replace('\n', '<br/>').encode('ascii', 'xmlcharrefreplace').decode('utf-8')
            story.append(Paragraph(formatted_html, code_style))
            story.append(PageBreak())

        doc.build(story)
        print(" -> 📄 Master PDF Blueprint generated successfully.")

    network_assets_dir = os.path.join(RAW_DIR, "network_assets")
    if os.path.exists(network_assets_dir):
        shutil.copytree(network_assets_dir, os.path.join(WORKSPACE_DIR, "Full_Playable_Site"), dirs_exist_ok=True)
        for root, dirs, files in os.walk(network_assets_dir):
            for file in files:
                src_path = os.path.join(root, file)
                ext = os.path.splitext(file)[1].lower()
                dest_dir = None

                if ext in ['.jpg', '.jpeg', '.png', '.svg', '.webp', '.gif', '.ico', '.avif']: dest_dir = "Global_Assets/Images"
                elif ext == '.css': dest_dir = "Global_Assets/CSS"
                elif ext == '.js': dest_dir = "Global_Assets/JS"
                elif ext in ['.woff', '.woff2', '.ttf', '.otf']: dest_dir = "Global_Assets/Fonts"

                if dest_dir:
                    dest_path = os.path.join(WORKSPACE_DIR, dest_dir, file)
                    if os.path.exists(dest_path):
                        name, extension = os.path.splitext(file)
                        dest_path = os.path.join(WORKSPACE_DIR, dest_dir, f"{name}_{hashlib.md5(src_path.encode()).hexdigest()[:6]}{extension}")
                    try: shutil.copy2(src_path, dest_path)
                    except: pass

    # Extract Copywriting Data
    html_path = os.path.join(RAW_DIR, "full_rendered_page.html")
    if os.path.exists(html_path):
        with open(html_path, "r", encoding="utf-8") as f: soup = BeautifulSoup(f, "html.parser")
        with open(os.path.join(WORKSPACE_DIR, "website_copywriting.json"), "w", encoding="utf-8") as f:
            json.dump({
                "headings": [h.get_text(strip=True) for h in soup.find_all(['h1', 'h2', 'h3']) if h.get_text(strip=True)],
                "paragraphs": [p.get_text(strip=True) for p in soup.find_all('p') if p.get_text(strip=True)]
            }, f, indent=4)

    print("✅ Workspace successfully organized.")

# ==========================================
# 5. EXPORT & ZIP
# ==========================================
def export_workspace():
    print(f"\n🗜️ PHASE 3: Zipping workspace...")
    zip_path = os.path.join(BASE_DIR, ZIP_FILENAME)
    shutil.make_archive(zip_path, 'zip', WORKSPACE_DIR)

    if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)

    print(f"\n🎉 DONE! Everything is packed into: {zip_path}.zip")

    # Auto-Download if running in Google Colab
    try:
        from google.colab import files
        print("📥 Google Colab detected: Triggering automatic browser download...")
        files.download(f"{ZIP_FILENAME}.zip")
    except ImportError:
        print("======================================================")
        print("Next Steps:")
        print(" 1. Extract the newly generated zip file.")
        print(" 2. Open 'LLM_Recreation_Blueprint.pdf'.")
        print(" 3. Drop the PDF into Claude 3.5 Sonnet to begin recreation.")
        print("======================================================\n")

# ==========================================
# MAIN RUN LOOP
# ==========================================
if __name__ == "__main__":
    if sys.platform == 'win32': asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        asyncio.run(extract_site(target_url))
        organize_data(target_url)
        export_workspace()
    except KeyboardInterrupt:
        print("\n\n❌ Extraction cancelled by user.")
    except Exception as e:
        print(f"\n❌ An error occurred: {str(e)}")

 🚀 AI-DRIVEN WEBSITE EXTRACTOR & PROMPT GENERATOR 
